# Hospital Readmission Prediction

This notebook builds a simple **logistic regression** model to predict whether a patient will be readmitted within 30 days.

The notebook uses the **Diabetes 130-US Hospitals for Years 1999-2008** dataset from Kaggle.

**Important:** This is an educational example, not a clinically validated model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)


## 2. Load the Kaggle dataset

The dataset is added directly to the Kaggle notebook as an input dataset.

The code automatically finds `diabetic_data.csv`, so you do not need to type the Kaggle folder name manually.

In [ ]:
import glob

# Find the Kaggle dataset file automatically
files = glob.glob('/kaggle/input/**/diabetic_data.csv', recursive=True)

print('Files found:')
print(files)

if len(files) == 0:
    raise FileNotFoundError(
        'diabetic_data.csv was not found. Add the Diabetes 130-US Hospitals dataset to your Kaggle notebook using Add Input.'
    )

data = pd.read_csv(files[0])

print('\nDataset loaded successfully!')
print('Dataset shape:', data.shape)
display(data.head())

## 3. Inspect the data

Before training, check the shape, data types, missing values, and target distribution.

In [ ]:
# The dataset uses '?' to represent missing values.
data = data.replace('?', np.nan)

print('Dataset shape:', data.shape)
print('\nData types:')
print(data.dtypes)
print('\nMissing values:')
print(data.isnull().sum())
print('\nOriginal readmission distribution:')
print(data['readmitted'].value_counts(dropna=False))

## 4. Create the target and separate features

The original `readmitted` column has three values:
- `<30`: readmitted within 30 days
- `>30`: readmitted after 30 days
- `NO`: not readmitted

For this binary classification problem, we create `readmitted_30_days`:
- `1`: readmitted within 30 days
- `0`: not readmitted within 30 days or readmitted after 30 days

In [ ]:
data['readmitted_30_days'] = (data['readmitted'] == '<30').astype(int)

print('Target distribution:')
print(data['readmitted_30_days'].value_counts())

print('\nTarget percentage:')
print(data['readmitted_30_days'].value_counts(normalize=True) * 100)

# Features available in the Kaggle dataset
numeric_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

categorical_features = [
    'race',
    'gender',
    'age',
    'change',
    'diabetesMed'
]

features = numeric_features + categorical_features

X = data[features]
y = data['readmitted_30_days']


## 5. Split into training and testing data

The model learns from the training set and is evaluated on unseen test data.

`stratify=y` keeps the proportion of readmitted and non-readmitted patients approximately similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=10,
    stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 6. Build the preprocessing pipeline

Numerical features are standardized so that their scales are comparable.
Categorical features are converted into numerical columns using one-hot encoding.

The preprocessing is placed inside a pipeline to avoid data leakage from the test set.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])


## 7. Create logistic regression with L2 regularization

`penalty='l2'` applies L2 regularization.

`C` is the inverse of regularization strength:
- Smaller `C` → stronger regularization
- Larger `C` → weaker regularization

The model will output probabilities, which are needed for ROC-AUC.

In [ ]:
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=0.8,
        max_iter=1000,
        random_state=10
    ))
])

# Train the complete pipeline.
model.fit(X_train, y_train)


## 8. Generate predictions

We obtain:
- Probabilities using `predict_proba()`
- Class predictions using `predict()`

ROC-AUC should be calculated using probabilities, not only 0/1 predictions.

In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print('First five predicted probabilities:', y_probability[:5])
print('First five class predictions:', y_pred[:5])


## 9. Evaluate using ROC-AUC

ROC-AUC measures how well the model ranks patients who are readmitted above patients who are not readmitted.

A value near 0.5 is similar to random ranking. Higher values indicate better discrimination.

In [ ]:
auc_score = roc_auc_score(y_test, y_probability)
print(f'ROC-AUC: {auc_score:.3f}')

false_positive_rate, true_positive_rate, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f'ROC-AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate / Recall')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()


## 10. Confusion matrix and classification report

A confusion matrix shows:
- True Positives: correctly predicted readmissions
- True Negatives: correctly predicted non-readmissions
- False Positives: predicted readmission, but no readmission occurred
- False Negatives: missed readmissions


In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No readmission', 'Readmission']
)
display.plot()
plt.title('Confusion Matrix')
plt.show()
